# VaryingVelocity — spatially varying velocity — 3D

`MagneticStandalone3D` · ASN 28 validation case (TRUST-LiMe).

## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``varying_velocity``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone3D` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

A strongly varying vortex velocity $u=(-y,x)$ is transferred onto the faces
from a functional field. With a uniform $b=b_0$, both transport terms vanish, so
$b$ must stay exactly $b_0$. This proves the varying-velocity path is correctly
instantiated and copied onto the face-centred unknown.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 7 | velocity is given data, frozen | exact invariant under a varying $u$ |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/varying_velocity.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = NZ0 = 17                        # baseline node counts (coarse)                       # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 17, "ny": 17, "nz": 17},            # the baseline mesh
    "standard": {"nx": 2*NX0-1, "ny": 2*NY0-1, "nz": 2*NZ0-1},
    "fine":     {"nx": 4*NX0-3, "ny": 4*NY0-3, "nz": 4*NZ0-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("varying_velocity.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition varying_velocity.data 2 -> PAR_varying_velocity.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
t, b = read_son(case.dataFileName_, "SONDE_CBX")
b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"varying-velocity invariance broken (rel. error {err:.3e})"
print("OK: uniform field invariant even under a varying velocity.")


## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`17×17×17`),
`standard` = cells ×2 (`33×33×33`), `fine` = cells ×4 (`65×65×65`).

Conservation of a uniform field under a varying prescribed velocity is a **discrete identity**: the scan asserts max\|b − b₀\|/\|b₀\| < 10⁻¹⁰ **at every level** — no order is reported.

In [ ]:
SONDE_TAG_SCAN = "SONDE_CBX"
B0 = 3.0

def read_son_dir(subdir, case_name, name):
    """(time, values) of probe `name` of case `case_name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def invariance_error(subdir, case_name):
    _, b = read_son_dir(subdir, case_name, SONDE_TAG_SCAN)
    return float(np.max(np.abs(b - B0)) / abs(B0))

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (17, 17, 17), "standard": (33, 33, 33), "fine": (65, 65, 65)}
scan_err = {lvl: invariance_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        max |b - b0|/b0")
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    print(f"{lvl:<9} {nx:>3}x{ny}x{nz:<3} {scan_err[lvl]:.3e}")
    assert scan_err[lvl] < 1e-10, f"varying-velocity invariance broken at {lvl}: {scan_err[lvl]:.3e}"
print("OK: uniform-field invariance under a varying velocity holds at every level.")

## Computer Performance


In [ ]:
run.tablePerf()